# 类型操作符与索引访问类型

学习目标：能从现有值和对象类型提取键、属性及数组元素类型，并用泛型保持键与返回值的对应关系。

前置知识：对象属性、数组、字面量联合、泛型函数与约束；能区分类型位置和值位置。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/12-type-operators/。

1. [main.ts](scripts/12-type-operators/main.ts)：配套实现与示例。
2. [tsconfig.json](scripts/12-type-operators/tsconfig.json)：本章独立项目配置。
3. [type-errors.ts](scripts/12-type-operators/type-errors.ts)、[tsconfig.errors.json](scripts/12-type-operators/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:12
```

Step 2：生成本章 JavaScript。

```bash
npm run build:12
```

Step 3：运行本章正常示例。

```bash
npm run run:12
# 正常退出；各段预期输出见代码注释。
```

正常片段均按正文顺序节选自 main.ts，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 keyof 描述允许使用的键

表单只允许选择课程对象已经声明的字段。keyof 接收对象类型，得到键的联合类型；它不是运行时枚举键的函数，也不会产生数组。

下面 CourseKey 只允许 title 和 hours。类型依赖 Course，新增字段后键集合会随之变化；无需另写一份容易遗漏的字符串联合。

```typescript
export type Course = { title: string; hours: number };
type CourseKey = keyof Course;
const selected: CourseKey = "title";
const course: Course = { title: "类型练习", hours: 4 };
console.log(selected, course[selected]);
// 预期输出：title 类型练习
```

## 2 索引签名与键的范围

有限对象的键可以是字面量联合，索引签名则表示一类键。数字索引签名的 keyof 为 number；字符串索引签名的 keyof 包含 string 和 number，因为数字属性访问可以对应字符串属性名。

这描述的是类型允许的访问范围，不保证每个数字或字符串位置都有实际属性。下面只观察确实写入的属性。

```typescript
type NumericSlots = { [index: number]: string };
type NamedSlots = { [key: string]: string };
const index: keyof NumericSlots = 3;
const numericName: keyof NamedSlots = 3;
const slots: NamedSlots = { "3": "已占用" };
console.log(index, slots[numericName]);
// 预期输出：3 已占用
```

## 3 typeof 查询静态类型

类型位置的 typeof 查询一个已有变量或其属性的静态类型。它不执行表达式，也不能直接对函数调用写类型查询；若需要函数的返回类型，可在后续工具类型中用 ReturnType。

下面 typeof defaults 得到对象的结构，而 console.log 中的 typeof 是 JavaScript 值运算，返回运行时类型标签。两者拼写相同，所在位置决定含义。

```typescript
const defaults = { theme: "light", retries: 2 };
type Settings = typeof defaults;
const settings: Settings = { theme: "dark", retries: 3 };
console.log(typeof defaults, settings.theme, settings.retries);
// 预期输出：object dark 3
```

## 4 T[K] 与联合索引

索引访问类型（indexed access type）写作 T[K]：T 代表被查询的对象类型，K 代表合法属性键的类型。它取出属性的类型，不是运行时读取对象。

当 K 是多个键的联合，结果是相应属性类型的联合，无法单凭这个结果恢复“哪个键对应哪个值”。变量名本身不是类型；使用值 key 的类型，需要写 typeof key。

```typescript
type Hours = Course["hours"];
type Cell = Course["title" | "hours"];
type AnyCell = Course[keyof Course];
const key = "hours";
const hours: Course[typeof key] = 6;
const cell: Cell = "进度";
const other: AnyCell = 8;
const duration: Hours = hours;
console.log(duration, cell, other);
// 预期输出：6 进度 8
```

## 5 从数组与元组提取元素类型

数组类型使用 number 作为索引类型可得到元素类型。先查询数组变量，再取元素，能让后续类型跟着数据结构更新。这里的 number 是类型，不是要访问某一个随机下标。

元组用固定下标可保留位置差异，用 number 则合并所有元素类型。这些类型查询不会执行数组读取，也不能证明运行时下标没有越界。

```typescript
const lessons = [{ title: "键", minutes: 15 }, { title: "索引", minutes: 20 }];
type Lesson = (typeof lessons)[number];
type Minutes = Lesson["minutes"];
const extra: Lesson = { title: "练习", minutes: 10 };
const minutes: Minutes = extra.minutes;
type Entry = readonly [string, number];
const first: Entry[0] = "练习";
const either: Entry[number] = 10;
console.log(first, minutes, either);
// 预期输出：练习 10 10
```

## 6 用 keyof 约束泛型参数

readField 中 T 代表传入对象的类型，K 代表本次选择的键类型。K extends keyof T 约束键必须属于这个对象；返回 T[K] 则把该键与它的属性类型联系起来。

调用方传入 hours 后，K 推断为这个字面量，因此返回值为 number，可以直接使用数值方法。若简单把返回值写成 string 与 number 的联合，调用方就会丢失这层关系。

```typescript
export function readField<T, K extends keyof T>(object: T, key: K): T[K] {
  return object[key];
}
console.log(readField(course, "hours").toFixed(1));
console.log(readField(course, "title").length > 0);
// 预期输出：4.0
// 预期输出：true
```

## 7 检查类型边界

下面的 [type-errors.ts](scripts/12-type-operators/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import { readField, type Course } from "./main.js";
const item: Course = { title: "键", hours: 1 };
readField(item, "missing"); // 对象没有这个键，不能满足 K 的约束。
type Missing = Course["missing"]; // 索引类型必须是已存在的属性键。
const field = "hours";
type WrongQuery = Course[field]; // field 是值，改用 typeof field。
const wrong: Course["hours"] = "1"; // 选出的是 number，不会转换字符串。
// 预期诊断包含：TS2345, TS2339, TS2538, TS2749, TS2322。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:12
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

keyof 提取键，类型位置的 typeof 查询已有值的类型，T[K] 查询属性类型；三者不会枚举或转换运行时数据。泛型键约束与返回类型共同保留调用关系。

## 练习

1. 为 Course 新增 published: boolean，更新正常输入；声明 Course[keyof Course] 的布尔值并确认通过检查。

2. 从本章 lessons 数组的元素提取 title 属性类型，命名为 Title；核对字符串可以赋给 Title，数值在独立反例中失败。

3. 把 readField 的键改成拼错的 hour，确认该调用得到类型诊断；恢复 hours 后核对输出 4.0。

### 提示

1. 除 Course 声明外，同步所有本练习中声明为 Course 的对象；不要把缺新字段误认成索引查询失败。
2. 先 typeof lessons，再用 number 取元素，最后用 "title" 取属性。
3. 只修改正常示例中读 hours 的那一次调用；诊断后恢复并重新生成。


### 参考解析

1. Course[keyof Course] 变为 string | number | boolean，true 可以赋值；原对象还需补 published 字段。
2. 可写 type Title = (typeof lessons)[number]["title"]，结果为 string；数值赋给 Title 在独立反例中应产生 TS2322。这个查询不运行数组索引。
3. hour 不在 keyof Course 中，调用应报告键实参不兼容；恢复 hours 后 readField 返回 number，toFixed(1) 得到 4.0。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [keyof：键与索引签名](https://www.typescriptlang.org/docs/handbook/2/keyof-types.html)；[typeof 与 Limitations](https://www.typescriptlang.org/docs/handbook/2/typeof-types.html)；[Indexed Access Types：联合、数组与值名边界](https://www.typescriptlang.org/docs/handbook/2/indexed-access-types.html)；[Using Type Parameters in Generic Constraints](https://www.typescriptlang.org/docs/handbook/2/generics.html#using-type-parameters-in-generic-constraints)。 |
